# Mask2Former Road Segmentation — Kaggle P100 / Google Colab

Trains **Mask2Former** (Swin Transformer backbone + deformable pixel decoder + masked cross-attention query decoder) on the DeepGlobe satellite road dataset.

| Model | Params | Road IoU (est.) | DeepGlobe F-score |
|---|---|---|---|
| DeepLabV3-ResNet50 | 42 M | 61–64 % | ~86 % |
| SegFormer-B2 | 25 M | 67–70 % | ~88 % |
| SegFormer-B5 | 82 M | 70–73 % | ~90 % |
| **Mask2Former-Swin-T** | **47 M** | **73–75 %** | **90.7 %** |

**Key differences from other notebooks:**
- **Label format is different** — each image must provide `class_labels` + `mask_labels`; the model runs Hungarian matching and computes its own loss
- **Loss is built-in** — `outputs.loss` = class CE + mask BCE + mask Dice, all bipartite-matched; no custom criterion needed
- **Validation needs semantic aggregation** — query predictions are aggregated into a semantic map for IoU
- **Training recipe** — AdamW + linear warmup + cosine decay, same as SegFormer

In [ ]:
import os

# ── Change to "colab" when running on Google Colab ────────────────────────────
PLATFORM = "kaggle"

if PLATFORM == "kaggle":
    OUTPUT_DIR      = "/kaggle/working"
    CHECKPOINT_PATH = "/kaggle/working/mask2former_checkpoint.pth"
    FINAL_MODEL     = "/kaggle/working/mask2former_road_final.pth"
    BEST_MODEL      = "/kaggle/working/mask2former_best.pth"
else:
    OUTPUT_DIR      = "/content"
    CHECKPOINT_PATH = "/content/mask2former_checkpoint.pth"
    FINAL_MODEL     = "/content/mask2former_road_final.pth"
    BEST_MODEL      = "/content/mask2former_best.pth"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Platform : {PLATFORM}  |  Output: {OUTPUT_DIR}")

In [ ]:
import subprocess, sys

# torch 2.6.0 + cu124:
#   - satisfies CVE-2025-32434 (torch.load needs >= 2.6)
#   - CUDA 12.4 retains sm_60 kernels for Kaggle P100
#   - cu121 index tops out at 2.5.1; cu124 is needed for 2.6.0
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124", "-q",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "kagglehub", "transformers>=4.35", "opencv-python", "scikit-image", "-q",
])
print("✅ Dependencies installed")

In [ ]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU      : {name}  ({vram:.1f} GB VRAM)")
    if "P100" in name:   print("✅ P100 — swin-t batch=4 at 512px")
    elif "T4" in name:   print("✅ T4   — swin-t batch=2 at 512px")
    elif "A100" in name: print("✅ A100 — swin-s batch=8 or swin-b batch=4")
else:
    raise RuntimeError("❌ No GPU found — enable GPU in accelerator settings.")

In [ ]:
import kagglehub

path = kagglehub.dataset_download("balraj98/deepglobe-road-extraction-dataset")
train_dir = os.path.join(path, "train")
if not os.path.isdir(train_dir):
    train_dir = path

sat_files = [f for f in os.listdir(train_dir) if f.endswith("_sat.jpg")]
print(f"Dataset  : {path}")
print(f"Images   : {len(sat_files)}")
assert len(sat_files) > 0, "No *_sat.jpg images found"

## Hyperparameters

| Backbone | Params | P100 batch | Est. min/epoch |
|---|---|---|---|
| swin-t | 47 M | 4 | ~25–35 |
| swin-s | 69 M | 4 | ~35–45 |
| swin-b | 107 M | 2 | ~50–65 |

In [ ]:
BACKBONE     = "swin-t"   # "swin-t" | "swin-s" | "swin-b"
IMG_SIZE     = 512
BATCH_SIZE   = 4          # P100/swin-t: 4.  P100/swin-b: 2.  T4/swin-t: 2.
NUM_EPOCHS   = 20         # Mask2Former converges slower than SegFormer
LR           = 1e-4
WEIGHT_DECAY = 0.05
WARMUP_FRAC  = 0.1
VAL_SPLIT    = 0.2
SEED         = 42
NUM_WORKERS  = 4
USE_CLAHE    = True
USE_AMP      = torch.cuda.is_available()
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

total_train     = int(len(sat_files) * (1 - VAL_SPLIT))
steps_per_epoch = total_train // BATCH_SIZE
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_FRAC)

print(f"Backbone : Mask2Former-{BACKBONE}")
print(f"IMG_SIZE : {IMG_SIZE}  BATCH={BATCH_SIZE}  EPOCHS={NUM_EPOCHS}")
print(f"LR       : {LR}  WD={WEIGHT_DECAY}  Warmup={warmup_steps}/{total_steps} steps")
print(f"AMP      : {USE_AMP}  Device: {DEVICE}")

In [ ]:
import cv2, random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF


def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    img_np = np.array(image)
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return Image.fromarray(cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2RGB))


class RoadSegmentationDataset(Dataset):
    def __init__(self, data_dir, file_list, img_size=512, augment=True, use_clahe=True):
        self.data_dir  = data_dir
        self.files     = sorted(file_list)
        self.augment   = augment
        self.use_clahe = use_clahe
        self.resize    = transforms.Resize((img_size, img_size))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name  = self.files[idx]
        image = Image.open(os.path.join(self.data_dir, name)).convert("RGB")
        mask  = Image.open(os.path.join(self.data_dir, name.replace("_sat.jpg", "_mask.png"))).convert("L")
        image, mask = self.resize(image), self.resize(mask)
        if self.use_clahe:
            image = apply_clahe(image)
        if self.augment:
            if random.random() > 0.5: image, mask = TF.hflip(image), TF.hflip(mask)
            if random.random() > 0.5: image, mask = TF.vflip(image), TF.vflip(mask)
            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                image, mask = TF.rotate(image, angle), TF.rotate(mask, angle)
        image = self.normalize(self.to_tensor(image))
        mask  = (self.to_tensor(mask) > 0.5).long().squeeze(0)
        return image, mask


all_files = sorted(f for f in os.listdir(train_dir) if f.endswith("_sat.jpg"))
rng = random.Random(SEED)
shuffled = all_files[:]
rng.shuffle(shuffled)
split       = int(len(shuffled) * (1 - VAL_SPLIT))
train_files, val_files = shuffled[:split], shuffled[split:]

train_ds = RoadSegmentationDataset(train_dir, train_files, IMG_SIZE, augment=True,  use_clahe=USE_CLAHE)
val_ds   = RoadSegmentationDataset(train_dir, val_files,   IMG_SIZE, augment=False, use_clahe=USE_CLAHE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} images  ({len(train_loader)} batches)")
print(f"Val  : {len(val_ds)} images  ({len(val_loader)} batches)")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import Mask2FormerForUniversalSegmentation

MODEL_IDS = {
    "swin-t": "facebook/mask2former-swin-tiny-ade-semantic",
    "swin-s": "facebook/mask2former-swin-small-ade-semantic",
    "swin-b": "facebook/mask2former-swin-base-ade-semantic",
}
print(f"Downloading {MODEL_IDS[BACKBONE]} ...")

model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL_IDS[BACKBONE],
    num_labels=2,                  # road / background
    ignore_mismatched_sizes=True,  # re-initialises class-prediction head for 2 classes
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Mask2Former-{BACKBONE} loaded  ({total_params:.1f} M params)")

## Label Formatter + Semantic Aggregation

Mask2Former needs labels in a different format from every other model — each image provides a list of class IDs and corresponding binary masks. The model matches queries to these masks via Hungarian algorithm, then computes its own combined loss.

Validation uses the reverse: we aggregate the 100 query predictions back into a per-pixel semantic map.

In [ ]:
def format_mask2former_labels(masks_batch, device):
    """
    Convert (B, H, W) class-index masks to the lists Mask2Former expects.

    Returns:
        class_labels : list[B] of 1-D long tensors — class IDs present in each image
        mask_labels  : list[B] of (num_classes, H, W) float tensors — binary masks
    """
    class_labels_list, mask_labels_list = [], []
    for mask in masks_batch:
        classes = torch.unique(mask).tolist()
        cls_t = torch.tensor(sorted(classes), dtype=torch.long, device=device)
        msk_t = torch.stack([(mask == c).float() for c in sorted(classes)]).to(device)
        class_labels_list.append(cls_t)
        mask_labels_list.append(msk_t)
    return class_labels_list, mask_labels_list


def semantic_predictions(model, imgs):
    """
    Aggregate query outputs into (B, H, W) argmax class predictions for IoU.
    """
    H, W    = imgs.shape[-2:]
    outputs = model(pixel_values=imgs)

    mask_logits = F.interpolate(
        outputs.masks_queries_logits, size=(H, W),
        mode="bilinear", align_corners=False,
    )  # (B, Q, H, W)

    # Drop no-object class (last column), compute per-query probabilities
    class_probs     = torch.softmax(outputs.class_queries_logits, dim=-1)[..., :-1]  # (B, Q, C)
    mask_probs      = torch.sigmoid(mask_logits)                                       # (B, Q, H, W)
    semantic_logits = torch.einsum("bqc,bqhw->bchw", class_probs, mask_probs)         # (B, C, H, W)
    return torch.argmax(semantic_logits, dim=1)                                        # (B, H, W)


print("✅ Label formatter and semantic aggregation helper ready")

In [ ]:
import math
from torch.optim import AdamW
from tqdm import tqdm

# Layer-wise LR: pretrained Swin encoder gets 10× lower rate than the new heads
backbone_params = [p for n, p in model.named_parameters() if "model.pixel_level_module.encoder" in n]
rest_params     = [p for n, p in model.named_parameters() if "model.pixel_level_module.encoder" not in n]

optimizer = AdamW([
    {"params": backbone_params, "lr": LR * 0.1},
    {"params": rest_params,     "lr": LR},
], weight_decay=WEIGHT_DECAY)


def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler   = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler      = torch.cuda.amp.GradScaler(enabled=USE_AMP)
global_step = 0


def save_checkpoint(epoch, model, optimizer, scheduler, scaler, val_iou, path):
    torch.save({
        "epoch": epoch, "global_step": global_step,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
        "val_iou": val_iou, "backbone": BACKBONE, "img_size": IMG_SIZE,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler, scaler):
    global global_step
    if not os.path.exists(path):
        return 0, 0.0
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    global_step = ckpt.get("global_step", 0)
    start = ckpt["epoch"] + 1
    best  = ckpt.get("val_iou", 0.0)
    print(f"✅ Resumed from epoch {start}  val IoU={best:.4f}")
    return start, best


def validate_epoch(model, loader):
    model.eval()
    ious, f1s = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                preds = semantic_predictions(model, imgs).cpu()
            for i in range(preds.size(0)):
                p  = preds[i].view(-1)
                t  = masks[i].view(-1)
                tp = ((p == 1) & (t == 1)).sum().item()
                fp = ((p == 1) & (t == 0)).sum().item()
                fn = ((p == 0) & (t == 1)).sum().item()
                ious.append(tp / (tp + fp + fn + 1e-7))
                prec = tp / (tp + fp + 1e-7)
                rec  = tp / (tp + fn + 1e-7)
                f1s.append(2 * prec * rec / (prec + rec + 1e-7))
    model.train()
    return float(sum(ious) / len(ious)), float(sum(f1s) / len(f1s))


# ── Run ───────────────────────────────────────────────────────────────────────
start_epoch, best_iou = load_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, scaler)

print(f"\n🚀 Mask2Former-{BACKBONE}  |  AMP={USE_AMP}  |  batch={BATCH_SIZE}  |  {IMG_SIZE}px")
print(f"   Epochs {start_epoch + 1} → {NUM_EPOCHS}  |  Warmup={warmup_steps} steps")
print("=" * 70)

model.train()
val_iou = best_iou

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_loss = 0.0
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", unit="batch")

    for i, (imgs, masks) in enumerate(bar):
        imgs = imgs.to(DEVICE)
        class_labels, mask_labels = format_mask2former_labels(masks, DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(
                pixel_values=imgs,
                class_labels=class_labels,
                mask_labels=mask_labels,
            )
            loss = outputs.loss   # CE(class) + BCE(mask) + Dice(mask), bipartite-matched

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        global_step += 1

        epoch_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{epoch_loss/(i+1):.4f}",
                        lr=f"{optimizer.param_groups[1]['lr']:.1e}")

    avg_loss        = epoch_loss / len(train_loader)
    val_iou, val_f1 = validate_epoch(model, val_loader)
    lr_now          = optimizer.param_groups[1]["lr"]

    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  "
          f"loss={avg_loss:.4f}  val_iou={val_iou:.4f}  val_f1={val_f1:.4f}  lr={lr_now:.2e}")

    save_checkpoint(epoch, model, optimizer, scheduler, scaler, val_iou, CHECKPOINT_PATH)

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch, "val_iou": val_iou,
            "backbone": BACKBONE, "img_size": IMG_SIZE,
        }, BEST_MODEL)
        print(f"   ⭐ New best IoU: {best_iou:.4f} — saved mask2former_best.pth")

print(f"\n✅ Training complete!  Best val IoU: {best_iou:.4f}")

In [ ]:
import zipfile

torch.save({
    "model_state_dict": model.state_dict(),
    "epoch": NUM_EPOCHS, "val_iou": val_iou,
    "backbone": BACKBONE, "img_size": IMG_SIZE,
}, FINAL_MODEL)
print(f"💾 Final model → {FINAL_MODEL}")
print(f"💾 Best model  → {BEST_MODEL}")

zip_file = os.path.join(OUTPUT_DIR, f"mask2former_{BACKBONE}_models.zip")
with zipfile.ZipFile(zip_file, "w") as zf:
    for f in [FINAL_MODEL, BEST_MODEL]:
        if os.path.exists(f):
            zf.write(f, os.path.basename(f))
print(f"📦 Zipped → {zip_file}")

if PLATFORM == "colab":
    from google.colab import files
    files.download(FINAL_MODEL)
    files.download(BEST_MODEL)
    files.download(zip_file)
else:
    print("\n📂 Kaggle: open Output panel → download files.")
    for f in [FINAL_MODEL, BEST_MODEL, zip_file]:
        if os.path.exists(f):
            print(f"   ✅ {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)")

---
## Using the trained model locally

```python
from models.architecture import build_mask2former, _Mask2FormerInferenceWrapper

raw = build_mask2former(backbone="swin-t")
ckpt = torch.load("mask2former_best.pth", map_location="cpu")
raw.load_state_dict(ckpt["model_state_dict"])
raw.eval()

# Wrap for run_inference.py / app.py — same {'out': (B,2,H,W)} interface
model = _Mask2FormerInferenceWrapper(raw)
```

---
## Note on loss magnitude

`outputs.loss` is the sum of three bipartite-matched terms (class CE + mask BCE + mask Dice). Its typical range is **2–5**, lower than the manually computed CE+Dice in the ResNet/SegFormer notebooks (range 3–8). A lower raw number does **not** mean faster convergence — always compare runs by **val IoU**, not loss magnitude.